# Predict low ratings with Kumo Engine: Netflix 5k

This notebook trains a graph neural network on a slice of the Netflix ratings data in Snowflake and writes its predictions back to Snowflake, all in-process with `kumo-engine`. For each movie with a high recent average rating, it predicts whether the movie will receive a low rating (below 3) in the next 60 days.

It reads the `Movies`, `Users`, and `Ratings` tables in `KUMO.NETFLIX_5K`; change `DATABASE` and `SCHEMA` in section 2 to use your own copy. Run it in a Snowflake Notebook on Container Runtime, or in any Python environment with a Snowflake `connections.toml` entry; see the [prerequisites](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/prerequisites) for supported versions. Training uses a GPU when one is available; CPU works for small data like this but is slow at production scale.

## 1. Install

Install Kumo Engine with the Snowflake connector, then restart the kernel so the new packages load.

In [ ]:
!pip install "kumo-engine[snowflake]"

In [ ]:
import torch

import kumo.engine as kumo

print('kumo.engine:', kumo.__version__)
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'none, training runs on CPU')

## 2. Connect

`SnowflakeConnector` reads the source tables and writes the predictions. Inside Snowpark Container Services, which includes a Snowflake Notebook on Container Runtime, `from_spcs_env()` uses the credentials the container already has. Elsewhere, `from_connection_toml()` reads your default Snowflake `connections.toml` entry, or the one you name with `connection_name=`. A connector addresses one warehouse, database, and schema: this cell takes the warehouse from the notebook in Snowflake, and from the `connections.toml` entry elsewhere.

In [ ]:
import os

from kumo_snowflake.spcs import SPCS_TOKEN_PATH

DATABASE, SCHEMA = 'KUMO', 'NETFLIX_5K'

if os.path.exists(SPCS_TOKEN_PATH):
    from snowflake.snowpark.context import get_active_session

    connector = kumo.SnowflakeConnector.from_spcs_env(
        warehouse=get_active_session().get_current_warehouse(),
        database=DATABASE, schema_name=SCHEMA)
    print('running in Snowpark Container Services')
else:
    connector = kumo.SnowflakeConnector.from_connection_toml(
        database=DATABASE, schema_name=SCHEMA)
    print('using connections.toml')

## 3. Read the catalog

`connector[name]` returns a `SourceTable`. Its columns, with their data types and semantic types, are read from the source on first access.

In [ ]:
print('tables:', connector.table_names())

for table_name in ('Movies', 'Users', 'Ratings'):
    for col in connector[table_name].columns:
        print(f'{table_name}.{col.name}', col.dtype,
              col.stype or '- (needs column statistics)')

## 4. Define the tables

`Table.from_source_table()` selects columns, sets the primary key and time column, and infers each column's semantic type from its data type and sampled values. `infer_metadata()` re-runs that inference for the table as defined, and `with_column()` returns a copy with one column's type set explicitly.

Set the types that inference can't know:

- `movieID` and `userID` are foreign keys. Inference sees one table at a time and judges them by their values, so declare them as `ID`.
- `rating` holds 1-5 stars. With only five distinct integer values it is inferred as categorical, but the query averages and compares ratings, so declare it numerical.
- `date` is stored as text in Snowflake. A time column must be a date, and declaring the type makes ingestion cast it.

In [ ]:
from kumoapi.typing import Dtype, Stype

movies = kumo.Table.from_source_table(
    connector['Movies'],
    ['MovieID', 'YearOfRelease'],
    primary_key='MovieID',
)
users = kumo.Table.from_source_table(connector['Users'], primary_key='userID')
ratings = (
    kumo.Table.from_source_table(connector['Ratings'], time_column='date')
    .infer_metadata()
    .with_column('movieID', stype=Stype.ID)
    .with_column('userID', stype=Stype.ID)
    .with_column('rating', stype=Stype.numerical)
    .with_column('date', dtype=Dtype.date)
)

for label, table in [('movies', movies), ('users', users),
                     ('ratings', ratings)]:
    resolved = ', '.join(f'{c.name}:{c.stype}' for c in table.definition.cols)
    print(f'{label:<8} {resolved}')

## 5. Assemble the graph

Each edge points from the table holding a foreign key to the table it references. `Graph.validate()` checks the graph's structure from the table schemas alone, before any data is ingested.

In [ ]:
from kumo.engine import Edge

graph = kumo.Graph(
    tables=[movies, users, ratings],
    edges=[
        Edge(src_table='Ratings', fkey='movieID', dst_table='Movies'),
        Edge(src_table='Ratings', fkey='userID', dst_table='Users'),
    ],
)
graph.validate()
print('graph is valid')

## 6. The predictive query

For each movie whose average rating over the past 60 days is above 3, predict whether it receives at least one rating below 3 in the next 60 days. `validate()` checks the query against the graph and prints any problems.

In [ ]:
QUERY = """
PREDICT COUNT_DISTINCT(Ratings.rating WHERE Ratings.rating < 3, 0, 60, days) > 0
FOR EACH Movies.MovieID
WHERE AVG(Ratings.rating, -60, 0, days) > 3
""".strip()
print(QUERY)

pquery = kumo.PredictiveQuery(graph=graph, query=QUERY)
pquery.validate()
print('\ntask type:', pquery.get_task_type())

## 7. Snapshot and generate a training table

`Graph.snapshot()` reads every table through its connector and builds an on-disk graph index under the scratch directory, `~/.cache/kumo-engine` by default; call `kumo.configure(scratch_dir=...)` first to use a larger disk. If the graph and its source data haven't changed, the existing snapshot is reused.

`generate_training_table(snapshot)` computes the training examples: the entities, anchor times, labels, and train, validation, and test splits.

In [ ]:
import pandas as pd

snapshot = graph.snapshot()
print('index_path:', snapshot.index_path)

train_table = pquery.generate_training_table(snapshot)
print('train_table_path:', train_table.train_table_path)
pd.read_parquet(train_table.train_table_path).head()

## 8. Train

`suggest_model_plan(snapshot)` proposes a model plan from the graph and the query; you can inspect and adjust it before training. `Trainer.fit()` trains on the training table's examples, with `snapshot` supplying the graph data, and returns the metrics and the location of the saved model bundle.

In [ ]:
trainer = kumo.Trainer(pquery.suggest_model_plan(snapshot))
result = trainer.fit(train_table, snapshot=snapshot)
print('metrics:', result.metrics)
print('\nbundle_dir:', result.bundle_dir)

On the full `NETFLIX_5K` tables, a well-trained model should reach a test AUROC of about 0.85 on this task.

## 9. Predict

`generate_prediction_table(snapshot)` selects the movies to score. For this query, it predicts as of the latest rating in the snapshot; to predict as of another time, pass `prediction_time=` a timezone-naive `datetime`, which is read as UTC.

`Trainer.predict()` scores the prediction table with the trained model. For binary classification, `binary_classification_threshold` sets the probability above which `TARGET_PRED` is `True`. With `output_config`, the predictions are also written to a Snowflake table, replacing its contents.

In [ ]:
from kumoapi.jobs import PredictionArtifactType, WriteMode

prediction_table = pquery.generate_prediction_table(snapshot)

output_config = kumo.PredictionOutputConfig(
    connector=connector,
    artifact_type=PredictionArtifactType.PREDICTIONS,
    table_name='NETFLIX_5K_PREDICTIONS_OUT',
    write_mode=WriteMode.OVERWRITE,
)

pred_result = trainer.predict(
    prediction_table,
    snapshot=snapshot,
    output_config=output_config,
    binary_classification_threshold=0.5,
)
print('rows:', len(pred_result.frame))
pred_result.frame.head()

In [ ]:
# The scores are an ordinary frame, so ranking is ordinary pandas.
top = pred_result.frame.sort_values('True_PROB', ascending=False).head(5)
print('Most likely to receive a low rating in the next 60 days:')
print(top[['ENTITY', 'True_PROB']].to_string(index=False))

## 10. Next steps

- Reload the trained model in a new session with `kumo.Trainer.load(result.bundle_dir)`, or publish it during training with `fit(..., export_config=...)`.
- Define a table from a read-only `SELECT` with `kumo.Table.from_sql(connector, ...)`.
- See the [package README](https://github.com/NVIDIA/kumo-engine/blob/main/packages/kumo-engine/README.md) for installation options and the full reference, and the [Kumo Engine documentation](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/overview) for prerequisites, GPU sizing, and more workflows.